<a href="https://colab.research.google.com/github/SukritiChauhan/cifar10-uncertainty-benchmark/blob/main/01_baseline_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

print(len(train_set), len(test_set))
x, y = train_set[0]
print(x.shape, y)

100%|██████████| 170M/170M [45:02<00:00, 63.1kB/s]


50000 10000
torch.Size([3, 32, 32]) 6


In [5]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./data /content/drive/MyDrive/cifar_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 64), block(64, 128), block(128, 256))
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(256 * 4 * 4, 10)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.fc(self.drop(x))

In [8]:
model = SmallCNN().to(device)
x, y = next(iter(train_loader))
out = model(x.to(device))
print(out.shape)
print(sum(p.numel() for p in model.parameters()))

torch.Size([128, 10])
1187274


In [9]:
def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

def train(model, epochs=15, lr=0.05):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader))
    for ep in range(epochs):
        model.train(); running = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step(); sched.step()
            running += loss.item()
        print(f"epoch {ep+1}: loss {running/len(train_loader):.3f}, "
              f"test acc {evaluate(model, test_loader):.3f}")

print("untrained accuracy:", evaluate(SmallCNN().to(device), test_loader))
model = SmallCNN()
train(model)
torch.save(model.state_dict(), "baseline.pt")

untrained accuracy: 0.1
epoch 1: loss 1.433, test acc 0.587
epoch 2: loss 0.994, test acc 0.709
epoch 3: loss 0.794, test acc 0.681
epoch 4: loss 0.670, test acc 0.744
epoch 5: loss 0.589, test acc 0.783
epoch 6: loss 0.528, test acc 0.811
epoch 7: loss 0.480, test acc 0.810
epoch 8: loss 0.440, test acc 0.806
epoch 9: loss 0.401, test acc 0.846
epoch 10: loss 0.360, test acc 0.849
epoch 11: loss 0.326, test acc 0.866
epoch 12: loss 0.286, test acc 0.885
epoch 13: loss 0.235, test acc 0.896
epoch 14: loss 0.196, test acc 0.901
epoch 15: loss 0.167, test acc 0.903


In [10]:
!cp baseline.pt /content/drive/MyDrive/
